# Downsampling experiment

Every model is scored on the same held-out meshes (`build_downsample_val_ds.py`),
so the curves differ only in how many specimens each model was trained on.

- **top panel** shape completion, chamfer distance vs training set size
- **bottom panel** classification, LOO and LOSO top-1 accuracy vs training set size

Chamfer comes from `<run>/shape_completion/evaluation/2phase_val_common_chamfer.csv`,
accuracy from the Table S3 written by `classification_tables.py`.

Run from `NSM/nsm/paper/`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

%matplotlib inline

# run directory : number of training specimens, smallest first
RUNS = {"../run_10spec": 10, "../run_30spec": 30, "../run_50spec": 50,
        "../run_70spec": 70, "../run_v72": 95}

S3      = "../downsample_tables/Table_S3_classification_stats.csv"
OUTDIR  = "../downsample_tables"
CHAMFER = "shape_completion/evaluation/2phase_val_common_chamfer.csv"

SPLIT   = "downsample_val_common"  # split tag the common-mesh evals wrote
LEVEL   = "position_20"            # fixed label space, so comparable across runs
LATENTS = "latent_opt"
SCALE   = 1e3                      # chamfer reported in units of 1e-3

COLORS = {"loo": "#1b4965", "specimen": "#bc4b51", "chamfer": "#386641"}

plt.rcParams.update({"font.sans-serif": "Liberation Sans", "font.size": 9,
                     "pdf.fonttype": 42, "figure.dpi": 120,
                     "axes.spines.top": False, "axes.spines.right": False})
os.makedirs(OUTDIR, exist_ok=True)

## Shape completion

Inner-joined on mesh name: a mesh any run failed on is dropped from all of them,
so the comparison stays paired. Watch the printed count — under 138 means some
run errored.

In [ ]:
for root, n in RUNS.items():
    p = os.path.join(root, CHAMFER)
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f"{n:3d} spec  rows={len(df):4d}  unique={df.mesh.nunique():4d}  "
              f"null_chamfer={df.chamfer.isna().sum():3d}")
    else:
        print(f"{n:3d} spec  MISSING")

In [ ]:
cols = {}
for root, n in RUNS.items():
    path = os.path.join(root, CHAMFER)
    if not os.path.exists(path):
        print(f"  missing {path} -- skipping")
        continue
    df = pd.read_csv(path)
    df["mesh"] = df["mesh"].str.replace("_partial.ply", "", regex=False)
    cols[n] = df.groupby("mesh")["chamfer"].mean() * SCALE

cham = pd.DataFrame(cols).dropna().sort_index(axis=1)
print(f"{len(cham)} meshes x {len(cham.columns)} runs")
cham.describe().T[["count", "mean", "std", "50%"]].rename(columns={"50%": "median"}).round(3)

## Classification

Reachable top-1 is the tabulated metric: under specimen masking a class whose
only specimen is hidden has no same-label gallery entry left, so its recall is
forced to 0 and it is excluded from the denominator instead.

In [ ]:
LEVELS_CHECK = ["broad_taxon", "family", "genus", "species", "position_20"]

d_all = pd.read_csv(S3)
d_all = d_all[(d_all["split"] == SPLIT) & (d_all["latents"] == LATENTS)
              & (d_all["level"].isin(LEVELS_CHECK))]
d_all = d_all.assign(n_spec=d_all["run"].map(sizes)).dropna(subset=["n_spec"])

# chance = guessing uniformly among the classes actually reachable in that run
k = d_all["n_reachable_classes"].replace(0, np.nan)
chance1 = 1 / k
chance5 = np.minimum(5, k) / k

# undefined where chance is already 1 (<=1 class for top-1, <=5 for top-5)
d_all["kappa1"] = ((d_all["top1_accuracy_reachable"] - chance1)
                   / (1 - chance1).replace(0, np.nan))
d_all["kappa5"] = ((d_all["top5_accuracy_reachable"] - chance5)
                   / (1 - chance5).replace(0, np.nan))

d_all.pivot_table(index=["level", "n_spec"], columns="eval_level",
                  values=["n_reachable_classes", "kappa1", "kappa5"]) \
     .reindex(LEVELS_CHECK, level="level").round(3)

## Figure

Median and IQR rather than mean and SD, because the chamfer distributions are
right-skewed and the Wilcoxon is a rank test — the plotted summary should match
the statistic. The chamfer axis is inverted so that better reads upward in both
panels.

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.2))
for lv, c in [("broad_taxon", "#386641"), ("family", "#8a6552"),
              ("genus", "#1b4965"), ("species", "#bc4b51")]:
    s = d_all[(d_all.level == lv) & (d_all.eval_level == "loo")].sort_values("n_spec")
    ax.plot(s["n_spec"], s["n_reachable_classes"], "o-", color=c, lw=1.8, ms=5, label=lv)
ax.set_xlabel("Training specimens"); ax.set_ylabel("Reachable classes")
ax.set_xticks(sorted(RUNS.values())); ax.legend(frameon=False)
fig.tight_layout()

In [ ]:
KAPPA_COLORS = {"loo": "#5fa8d3", "specimen": "#e3a09e"}

LEVEL_TITLES = {"position_20": "Normalized spinal position (20% bins)",
                "broad_taxon": "Broad taxon",
                "family": "Family",
                "genus": "Genus",
                "species": "Species"}

fig, axes = plt.subplots(2, 2, figsize=(8.5, 6.5), sharex=True)

sizes_x = list(cham.columns)
mean = np.array([cham[n].mean() for n in sizes_x])
sd   = np.array([cham[n].std(ddof=1) for n in sizes_x])
med  = [cham[n].median() for n in sizes_x]
q1   = [cham[n].quantile(0.25) for n in sizes_x]
q3   = [cham[n].quantile(0.75) for n in sizes_x]

ax = axes[0, 0]
ax.fill_between(sizes_x, mean - sd, mean + sd, color=COLORS["chamfer"],
                alpha=0.12, lw=0)
ax.fill_between(sizes_x, q1, q3, color=COLORS["chamfer"], alpha=0.18, lw=0)
ax.plot(sizes_x, mean, "o--", color=COLORS["chamfer"], lw=1.4, ms=4,
        mfc="white", label="mean $\\pm$ SD")
ax.plot(sizes_x, med, "o-", color=COLORS["chamfer"], lw=1.8, ms=5,
        label="median [IQR]")
ax.set_title(f"Shape completion", loc="left")
ax.set_ylabel("Chamfer ($\\times10^{-3}$)")
ax.legend(frameon=False, fontsize=7.5, loc="upper right")

def accuracy_panel(ax, lv, title=None):
    g = d_all[d_all["level"] == lv]
    for ev, label in [("loo", "LOO"), ("specimen", "LOSO")]:
        s = g[g["eval_level"] == ev].sort_values("n_spec")
        if s.empty:
            continue
        ax.plot(s["n_spec"], s["top1_accuracy_reachable"] * 100, "o-",
                color=COLORS[ev], lw=1.8, ms=5, label=f"{label} top-1")
        ax.plot(s["n_spec"], s["top5_accuracy_reachable"] * 100, "o--",
                color=COLORS[ev], lw=1.3, ms=4, mfc="white", alpha=0.8,
                label=f"{label} top-5")
        ax.plot(s["n_spec"], s["kappa1"] * 100, "s-",
                color=KAPPA_COLORS[ev], lw=1.6, ms=4,
                label=f"{label} top-1 (chance-corr.)")
    name = title or LEVEL_TITLES.get(lv, lv)
    ax.set_title(f"{name}",
                 loc="left")
    ax.set_ylim(0, 100)

accuracy_panel(axes[0, 1], "position_20")
accuracy_panel(axes[1, 0], "broad_taxon")
accuracy_panel(axes[1, 1], "family")

axes[0, 1].legend(frameon=False, fontsize=6.5, loc="lower right", ncol=2)
for ax in (axes[0, 1], axes[1, 0]):
    ax.set_ylabel("Accuracy (%)")
for ax in axes[1, :]:
    ax.set_xlabel("Training specimens")
for ax in axes.ravel():
    ax.set_xticks(sorted(RUNS.values()))

fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "Fig_downsample_levels.pdf"), dpi=300)
plt.show()

## Paired Wilcoxon signed-rank

The runs share their meshes, so each is tested against the largest run on the
per-mesh differences rather than as an independent sample. `median_diff` is the
median of those differences, not the difference of the medians; negative means
lower chamfer, i.e. better than the reference.

In [ ]:
def holm(p):
    """Holm-Bonferroni adjusted p-values, same order as the input."""
    p = np.asarray(p, dtype=float)
    n = len(p)
    order = np.argsort(p)
    adj, running = np.empty(n), 0.0
    for i, k in enumerate(order):
        running = max(running, (n - i) * p[k])   # enforce monotonicity
        adj[k] = min(running, 1.0)
    return adj


def paired(a, b, label):
    """a vs b, both columns of cham. Positive median_diff = a worse than b."""
    x, y = cham[a], cham[b]
    d = x - y
    res = stats.wilcoxon(x, y)
    return {"comparison": label, "n_pairs": len(d),
            "median_a": float(x.median()), "median_b": float(y.median()),
            "median_diff": float(np.median(d)),
            "pct_change": float(100 * np.median(d) / y.median()),
            "statistic": float(res.statistic), "p_value": float(res.pvalue)}


sizes = list(cham.columns)
ref = max(sizes)

# deficit: each run against the full training set
vs_full = pd.DataFrame([paired(n, ref, f"{n} vs {ref} spec")
                        for n in sizes if n != ref])

# marginal: each step up the series
steps = pd.DataFrame([paired(lo, hi, f"{lo} -> {hi} spec")
                      for lo, hi in zip(sizes, sizes[1:])])

# Holm correction within each family of tests, not across both
for t in (vs_full, steps):
    t["p_holm"] = holm(t["p_value"])

print("Deficit vs full training set")
print(vs_full.round(4).to_string(index=False))
print("\nMarginal gain per step")
print(steps.round(4).to_string(index=False))

## Write the stats file

In [ ]:
stats_path = os.path.join(OUTDIR, "downsample_stats.txt")
with open(stats_path, "w") as fh:
    fh.write(f"Downsampling experiment, {len(cham)} shared held-out meshes\n")
    fh.write("=" * 72 + "\n\n")

    fh.write("Chamfer distance (x1e-3) by training set size\n")
    fh.write(cham.describe().T[["count", "mean", "std", "50%"]]
             .rename(columns={"50%": "median"}).round(3).to_string() + "\n\n")

    fh.write("Paired Wilcoxon signed-rank, deficit vs the largest training set\n")
    fh.write("(negative median_diff = lower chamfer = better than reference)\n")
    fh.write(vs_full.round(4).to_string(index=False) + "\n\n")

    fh.write("Paired Wilcoxon signed-rank, marginal gain per step\n")
    fh.write("(positive median_diff = the smaller run is worse)\n")
    fh.write("Steps are not independent; read as descriptive.\n")
    fh.write(steps.round(4).to_string(index=False) + "\n\n")

    fh.write("Holm-adjusted within each family of tests separately.\n\n")

    fh.write(f"Classification, reachable accuracy, {SPLIT}\n")
    fh.write(d_all[["run", "n_spec", "level", "eval_level", "n_eligible",
                    "n_reachable_classes", "top1_accuracy_reachable",
                    "top5_accuracy_reachable", "kappa1"]]
             .sort_values(["level", "n_spec", "eval_level"])
             .round(4).to_string(index=False) + "\n")

print(f"wrote {stats_path}")
print(open(stats_path).read())